In [ ]:
!pip install "transformers<5.0.0"
!pip install setfit
# wandb api key : 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

In [ ]:
import pandas as pd
from setfit import SetFitModel, SetFitTrainer
from datasets import Dataset
from sentence_transformers.losses import CosineSimilarityLoss # Import the loss class

df = pd.read_csv("./data.txt", sep="|", names=["text", "label"], engine="python")
df['text'] = df['text'].str.strip()
df['label'] = df['label'].str.strip()

print(f"Loaded {len(df)} examples.")
print(df.head())

dataset = Dataset.from_pandas(df)
model = SetFitModel.from_pretrained("sentence-transformers/paraphrase-mpnet-base-v2")
trainer = SetFitTrainer(
    model=model,
    train_dataset=dataset,
    loss_class=CosineSimilarityLoss, # Pass the class object, not the string
    batch_size=16,
    num_iterations=20, # The magic number for SetFit (generates pairs)
    column_mapping={"text": "text", "label": "label"}
)

print("Training... (This takes about 30 seconds on a laptop)")
trainer.train()

model.save_pretrained("./my_setfit_model")
print("Model saved to ./my_setfit_model")

print("\n--- Testing SetFit Model ---")
test_tasks = [
    "The production database is down and customers cannot login", # Should be High
    "Order more coffee for the break room",                      # Should be Low
    "Schedule a check-in call with the client",                  # Should be Medium
    "Fix the typo in the footer"                                 # Should be Low
]

id2label = {0: "Low", 1: "Medium", 2: "High"}

preds = model(test_tasks)
for task, pred in zip(test_tasks, preds):
    print(f"Task: {task} \n -> Prediction: {pred}\n")

In [ ]:
# Smaller dataset
from setfit import SetFitModel

print("Loading model...")
model = SetFitModel.from_pretrained("./my_setfit_model")

id2label = {0: "Low", 1: "Medium", 2: "High"}

test_tasks = [
    "The production database is down and customers cannot login",
    "Order more coffee for the break room",
    "Schedule a check-in call with the client",
    "Fix the typo in the footer",
    "The server is on fire",
    "Plan the holiday party",
    "API response time is too slow",
    "Database down"
]

print("\n--- Model Predictions ---")
preds = model(test_tasks)

for task, pred in zip(test_tasks, preds):
    print(f"Task: '{task}' \n  -> Priority: {pred}\n")

In [ ]:
from setfit import SetFitModel
import torch

class TaskPriorityModel:
    def __init__(self, model_path="./my_setfit_model"):
        self.model = SetFitModel.from_pretrained(model_path)
        self.id2label = {0: "Low", 1: "Medium", 2: "High"}

    def predict(self, text):
        probs = self.model.predict_proba([text])[0]
        confidence_score, pred_id = torch.max(probs, dim=0)
        confidence_score = confidence_score.item() 
        if confidence_score < 0.60:
            return "Needs Review", confidence_score
        return self.id2label[pred_id.item()], confidence_score

if __name__ == "__main__":
    ai = TaskPriorityModel()
    priority, conf = ai.predict("Server is down")
    print(f"Server is down: {priority} ({conf:.1%})") 
    priority, conf = ai.predict("Check it")
    print(f"Check it: {priority} ({conf:.1%})")